# Step 3 — Reusable Code

En este notebook se valida que el flujo de Machine Learning ya fue modularizado en scripts reutilizables.

Los stages son:

1. `prepare_data.py`
2. `split_data.py`
3. `train.py`
4. `evaluate.py`

El objetivo es confirmar que el pipeline modular reproduce las mismas métricas obtenidas en el prototipo de Fase 1.

# Imports

In [3]:
# IMPORTS

from pathlib import Path
import subprocess
import sys
import json
import yaml
import pandas as pd

# Config

In [5]:
# CONFIG

PROJECT_ROOT = Path.cwd()
PARAMS_PATH = PROJECT_ROOT / "params.yaml"

with open(PARAMS_PATH, "r", encoding="utf-8") as conf_file:
    config = yaml.safe_load(conf_file)

# Usamos el mismo Python del kernel activo para evitar problemas de entorno
PYTHON_EXECUTABLE = sys.executable

print("Project root:", PROJECT_ROOT)
print("Python executable:", PYTHON_EXECUTABLE)
config

Project root: C:\Users\daniel.martinez\real_state_price_predictor\tog_dme
Python executable: C:\Users\daniel.martinez\AppData\Local\anaconda3\python.exe


{'base': {'numpy_seed': 33},
 'data_load': {'raw_data_path': 'data/raw/Guadalajara 4Q22.xlsx'},
 'prepare': {'prepared_data_path': 'data/processed/prepared_data.csv',
  'delivery_base_date': '2022-09-01',
  'towns_to_drop': ['Chapala', 'El Salto'],
  'price_per_sqm_max': 90000,
  'sqm_max': 250.0,
  'classification_mapping': {'S': -2, 'E': -1, 'M': 0, 'R': 1, 'RP': 2},
  'dead_columns': ['id',
   'project_name',
   'address',
   'promoter',
   'alcoba',
   'room_serv',
   'first_price',
   'initial_date',
   'entry_date',
   'update_date',
   'fin_op',
   'inventory_months'],
  'final_drop_columns': ['colony',
   'town',
   'price_per_sqm',
   'comm_succ',
   'absortion',
   'sold_units',
   'baths']},
 'split': {'test_size': 0.2,
  'random_state': 42,
  'train_data_path': 'data/processed/train_data.csv',
  'test_data_path': 'data/processed/test_data.csv'},
 'train': {'model_type': 'ridge',
  'scoring': 'r2',
  'cv_splits': 10,
  'cv_random_state': 42,
  'n_jobs': -1,
  'alphas': [1000

## 1. Validar que existen los scripts

In [7]:
# CHECK STAGE FILES

stage_files = [
    "src/stages/prepare_data.py",
    "src/stages/split_data.py",
    "src/stages/train.py",
    "src/stages/evaluate.py",
]

for stage_file in stage_files:
    path = PROJECT_ROOT / stage_file
    print(stage_file, "->", path.exists())

    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {stage_file}")

src/stages/prepare_data.py -> True
src/stages/split_data.py -> True
src/stages/train.py -> True
src/stages/evaluate.py -> True


## 2. Ejecutar stages reutilizables

In [9]:
# RUN PIPELINE STAGES

stages = [
    "src/stages/prepare_data.py",
    "src/stages/split_data.py",
    "src/stages/train.py",
    "src/stages/evaluate.py",
]

for stage in stages:
    print("=" * 80)
    print(f"Running stage: {stage}")
    print("=" * 80)

    result = subprocess.run(
        [PYTHON_EXECUTABLE, stage],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Falló el stage: {stage}")

print("Pipeline modular ejecutado correctamente.")

Running stage: src/stages/prepare_data.py
Prepared data saved to: data/processed/prepared_data.csv
Prepared data shape: (536, 18)

Running stage: src/stages/split_data.py
Prepared data loaded from: data/processed/prepared_data.csv
Prepared data shape: (536, 18)
Train data saved to: data/processed/train_data.csv
Test data saved to: data/processed/test_data.csv
Train shape: (428, 18)
Test shape: (108, 18)
Scaler X saved to: models/scaler_X.pkl
Scaler Y saved to: models/scaler_Y.pkl
Feature names saved to: models/feature_names.json

Running stage: src/stages/train.py
Train data loaded from: data/processed/train_data.csv
Train data shape: (428, 18)
Modelo entrenado correctamente.
Mejores hiperparámetros: {'alpha': 20}
R2 CV train: 0.8533903044819209
Modelo guardado en: models/modelo_final.pkl

Running stage: src/stages/evaluate.py
Train data loaded from: data/processed/train_data.csv
Test data loaded from: data/processed/test_data.csv
Train shape: (428, 18)
Test shape: (108, 18)
Model load

## 3. Validar archivos generados

In [11]:
# VALIDATE OUTPUT FILES

expected_outputs = [
    config["prepare"]["prepared_data_path"],
    config["split"]["train_data_path"],
    config["split"]["test_data_path"],
    config["artifacts"]["model_path"],
    config["artifacts"]["scaler_x_path"],
    config["artifacts"]["scaler_y_path"],
    config["artifacts"]["feature_names_path"],
    config["metrics"]["metrics_path"],
]

for output in expected_outputs:
    path = PROJECT_ROOT / output
    print(output, "->", path.exists())

    if not path.exists():
        raise FileNotFoundError(f"No se generó el archivo esperado: {output}")

data/processed/prepared_data.csv -> True
data/processed/train_data.csv -> True
data/processed/test_data.csv -> True
models/modelo_final.pkl -> True
models/scaler_X.pkl -> True
models/scaler_Y.pkl -> True
models/feature_names.json -> True
reports/metrics.json -> True


## 4. Revisar datasets generados

In [13]:
# CHECK GENERATED DATASETS

prepared_data = pd.read_csv(config["prepare"]["prepared_data_path"])
train_data = pd.read_csv(config["split"]["train_data_path"])
test_data = pd.read_csv(config["split"]["test_data_path"])

print("Prepared data shape:", prepared_data.shape)
print("Train data shape:", train_data.shape)
print("Test data shape:", test_data.shape)

prepared_data.head()

Prepared data shape: (536, 18)
Train data shape: (428, 18)
Test data shape: (108, 18)


,classification,sqm,terrace,bhk,park_u,levels,price,months_in_sale,master_plan_units,total_units,inventory,months_to_delivery,Guadalajara,Jocotepec,Tlajomulco de Zúñiga,Tlaquepaque,Tonalá,Zapopan
0,1,222.0,0.0,3.0,2.0,11.0,9077955.0,85.150685,81,44,4,0.0,True,False,False,False,False,False
1,1,111.0,0.0,2.0,2.0,11.0,5922418.0,85.150685,81,44,4,0.0,True,False,False,False,False,False
2,1,110.0,0.0,2.0,2.0,11.0,5867088.0,85.150685,81,44,4,0.0,True,False,False,False,False,False
3,1,91.0,0.0,2.0,2.0,10.0,5300000.0,84.295890,392,392,8,0.0,False,False,False,False,False,True
4,1,131.0,0.0,3.0,2.0,10.0,6503000.0,84.295890,392,392,8,0.0,False,False,False,False,False,True


## 5. Validar métricas finales

In [15]:
# LOAD METRICS

with open(config["metrics"]["metrics_path"], "r", encoding="utf-8") as f:
    metrics = json.load(f)

metrics

{'R2_train': 0.8722779520936734,
 'R2_test': 0.8586681107871221,
 'RMSE_train_pesos': 820764.7607973475,
 'RMSE_test_pesos': 724228.728716661,
 'MAE_train_pesos': 598614.5921648004,
 'MAE_test_pesos': 549476.6093178215}

## 6. Comparar métricas contra fase 1

In [17]:
# COMPARE AGAINST PHASE 1 RESULTS

expected_metrics = {
    "R2_test": 0.8586681107871221,
    "RMSE_test_pesos": 724228.7287166608,
    "MAE_test_pesos": 549476.6093178215,
}

print("Comparación contra monolito de Fase 1:")
print("-" * 60)

for metric_name, expected_value in expected_metrics.items():
    current_value = metrics[metric_name]
    difference = abs(current_value - expected_value)

    print(f"{metric_name}")
    print(f"  Esperado: {expected_value}")
    print(f"  Actual:   {current_value}")
    print(f"  Dif:      {difference}")
    print()

# Tolerancias razonables por precisión decimal
assert abs(metrics["R2_test"] - expected_metrics["R2_test"]) < 1e-8
assert abs(metrics["RMSE_test_pesos"] - expected_metrics["RMSE_test_pesos"]) < 1e-2
assert abs(metrics["MAE_test_pesos"] - expected_metrics["MAE_test_pesos"]) < 1e-2

print("Métricas validadas correctamente contra Fase 1.")

Comparación contra monolito de Fase 1:
------------------------------------------------------------
R2_test
  Esperado: 0.8586681107871221
  Actual:   0.8586681107871221
  Dif:      0.0

RMSE_test_pesos
  Esperado: 724228.7287166608
  Actual:   724228.728716661
  Dif:      2.3283064365386963e-10

MAE_test_pesos
  Esperado: 549476.6093178215
  Actual:   549476.6093178215
  Dif:      0.0

Métricas validadas correctamente contra Fase 1.


## Conclusión

El flujo modular reproduce correctamente el resultado del prototipo original.

Esto confirma que el proyecto ya pasó de un notebook monolítico a scripts reutilizables, manteniendo las mismas métricas del modelo ganador.